# N1 · flash attention: 内存从 O(S²) 降到 O(S)

> 复用 `src/capstone_attn_speedup.py` · flash attention 用 tiling + online softmax (接 cuda-essentials),
> 不存 S×S 分数矩阵 → 内存线性。看内存随序列长度的对比。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import capstone_attn_speedup as ca
print('kernel-engineering src 就绪')

## 1. naive (存全矩阵 O(S²)) vs flash (tiling O(S)) 的内存

In [ ]:
import matplotlib, matplotlib.pyplot as plt
matplotlib.rcParams['axes.unicode_minus']=False
for f in ['Microsoft YaHei','SimHei','DejaVu Sans']:
    try: matplotlib.rcParams['font.sans-serif']=[f]; break
    except Exception: pass
rows = ca.speedup_curve()
seq = [r['seq_len'] for r in rows]
naive = [r['naive_mb'] for r in rows]
flash = [r['flash_mb'] for r in rows]
sp = [r['speedup'] for r in rows]
print(f"{'序列长':>8} {'naive MB':>10} {'flash MB':>10} {'省内存':>8}")
for r in rows: print(f"{r['seq_len']:>8} {r['naive_mb']:>10} {r['flash_mb']:>10} {r['speedup']:>7}x")
plt.figure(figsize=(7,4.3))
plt.plot(seq, naive, 'o-', label='naive (O(S²) 存全矩阵)', color='C3')
plt.plot(seq, flash, 's-', label='flash (O(S) tiling)', color='C0')
plt.xlabel('序列长度 S'); plt.ylabel('峰值内存 (MB)'); plt.legend()
plt.title('flash attention: 内存 O(S²)→O(S) (不存 S×S 分数矩阵)'); plt.tight_layout(); plt.show()
print('→ naive 内存随 S² 爆炸; flash 用 tiling+online softmax 线性 → 长序列可行 (本专题核心)。')

## 2. 反思
- flash attention = tiling (分块) + online softmax (cuda-essentials) → 不物化 S×S 矩阵。
- 内存 O(S²)→O(S), 是长上下文 (long-context) 可行的关键 kernel。
- 本专题其余 src (`fused_mlp`/`rmsnorm_kernel`/`triton_style`/`cutlass_layout`) 是 kernel 融合/布局的模拟。